# Lab 6 — [Assigned County Name]

Complete the practice and Q0, then Q1, Q2 and Q3 for your assigned county. Replace bracketed text and SQL/plot placeholders with your own work.

## Assigned County

**County/city:** [your assigned official name]

**FIPS:** [the code you looked up yourself or with Gemini]

Use the same name and FIPS recorded in lab6.sql.

In [ ]:
AREA_NAME = "REPLACE WITH YOUR ASSIGNED COUNTY OR CITY NAME"

## Setup and Database Practice
### Enable Secrets in this new notebook
Open Colab Secrets (key icon). Enable **Notebook access** for `DB_HOST`, `DB_NAME`, `DB_USER`, and `DB_PASSWORD`.
Existing Secret values are reused, but access is granted to this notebook separately. **CENSUS_API_KEY is not used.**
Do not display the values. Use only the existing lab database.

In [ ]:
%pip -q install psycopg2-binary

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import userdata

In [ ]:
try:
    conn = psycopg2.connect(
        host=userdata.get("DB_HOST"), port=5432,
        dbname=userdata.get("DB_NAME"), user=userdata.get("DB_USER"),
        password=userdata.get("DB_PASSWORD"),
        sslmode="require", connect_timeout=10
    )
except psycopg2.Error:
    raise RuntimeError("Connection failed. Check DB Secrets and instance access privately.") from None
cur = conn.cursor()
print("Connected.")

In [ ]:
def read_query(sql):
    cur.execute(sql)
    rows = cur.fetchall()
    columns = [col[0] for col in cur.description]
    return pd.DataFrame(rows, columns=columns)

### Look up the FIPS from the assigned name
Use the official name, including `County` or `city`. Confirm one match instead of selecting an arbitrary first row.
Your Q1–Q3 SQL must filter this same FIPS.

In [ ]:
cur.execute("SELECT fips, name FROM public.name WHERE name = %s;", (AREA_NAME,))
matches = cur.fetchall()
assert len(matches) == 1, "Expected one county-name match. Check the spelling or ask the instructor."
ASSIGNED_FIPS, AREA_NAME = matches[0]
print(AREA_NAME, ASSIGNED_FIPS)
conn.commit()

### INSERT — one fake name
First confirm FIPS 51999 is unused. All successful practice writes affect this fake row only. Do not add population or income observations.

In [ ]:
cur.execute("SELECT fips, name FROM public.name WHERE fips = '51999';")
assert cur.fetchone() is None, "51999 already exists. Inspect it before proceeding."
conn.commit()
cur.execute("INSERT INTO public.name (fips, name) VALUES ('51999', 'Practice County');")
conn.commit()
cur.execute("SELECT fips, name FROM public.name WHERE fips = '51999';")
print(cur.fetchall())

### UPDATE — change only the fake name
Expected: 51999 / Practice County - Updated.

In [ ]:
cur.execute("UPDATE public.name SET name = 'Practice County - Updated' WHERE fips = '51999';")
conn.commit()
cur.execute("SELECT fips, name FROM public.name WHERE fips = '51999';")
print(cur.fetchall())

### DELETE — remove the fake name
Expected: `[]` means that the SELECT returns no fake row. It does not mean the table was deleted.

In [ ]:
cur.execute("DELETE FROM public.name WHERE fips = '51999';")
conn.commit()
cur.execute("SELECT fips, name FROM public.name WHERE fips = '51999';")
print(cur.fetchall())
conn.commit()

### A rejected INSERT and rollback

The NULL FIPS is rejected. After `conn.rollback()`, the same open connection can run another query. The following SELECT returns the Fairfax name without reconnecting.

In [ ]:
try:
    cur.execute("""INSERT INTO public.name (fips, name)
                   VALUES (NULL, 'Practice County');""")
except psycopg2.errors.NotNullViolation:
    conn.rollback()
    print("Rejected NULL FIPS. Transaction reset; connection stays open.")
else:
    conn.rollback()
    raise RuntimeError("Expected primary-key rejection. Ask the instructor.")

cur.execute("SELECT name FROM public.name WHERE fips = '51059';")
print(cur.fetchall())
conn.commit()

### Q0–Q3 are read-only
The fake row is gone. The remaining cells read the stored source tables without changing them.

In [ ]:
conn.set_session(readonly=True, autocommit=True)
print("Ready for read-only analysis.")

## Q0 — Guided Example

### Question

What were the 2024 population estimates for the 30 covered Virginia county-level areas?

This class example uses all covered areas. Your own Q1–Q3 use only your assigned county.

### SQL

In [ ]:
sql_q0 = """
SELECT n.fips, n.name, p.year, p.population
FROM public.name AS n
JOIN public.population AS p ON n.fips = p.fips
WHERE n.fips LIKE '51%' AND p.year = 2024
ORDER BY p.population DESC, n.fips;
"""

### Python/Pandas

In [ ]:
cur.execute(sql_q0)
rows = cur.fetchall()
columns = [col[0] for col in cur.description]
q0 = pd.DataFrame(rows, columns=columns)
display(q0)
print("Covered areas:", len(q0))

### Visualization

In [ ]:
plot_q0 = q0.sort_values(["population", "fips"])
ax = plot_q0.plot.barh(x="name", y="population", legend=False, figsize=(9, 10))
ax.set_title("2024 population: the 30 covered Virginia areas")
ax.set_xlabel("Population (people)")
ax.set_ylabel("County or county-equivalent city")
ax.ticklabel_format(style="plain", axis="x")
plt.tight_layout()
plt.show()

### Interpretation

Fairfax County has the largest population estimate among these 30 covered areas, and York County has the smallest. This chart compares the covered areas, not a statewide population total.

## Q1 — Population Growth

### Question

For [MY COUNTY], what was the annual population growth rate for each available year in 2015–2024, compared with the previous calendar year?

### SQL

In [ ]:
sql_q1 = """
PASTE_YOUR_Q1_SQL_HERE
"""

### Python/Pandas

In [ ]:
q1 = read_query(sql_q1)
display(q1)

### Visualization

In [ ]:
# Plot q1 here. Gemini may help you write this code.
# Use your assigned county, correct years, metric, and units.

### Interpretation

[Write 1–2 sentences answering Q1 from your table/chart. Describe the population pattern and use a relevant year/value.]

## Q2 — Household-Income Growth

### Question

For [MY COUNTY], what was the annual growth rate of nominal median household income for each available year in 2015–2024, compared with the previous calendar year?

### SQL

In [ ]:
sql_q2 = """
PASTE_YOUR_Q2_SQL_HERE
"""

### Python/Pandas

In [ ]:
q2 = read_query(sql_q2)
display(q2)

### Visualization

In [ ]:
# Plot q2 here. Gemini may help you write this code.
# Use your assigned county, correct years, metric, and units.

### Interpretation

[Write 1–2 sentences answering Q2 from your table/chart. Describe the income pattern and use a relevant year/value.]

## Q3 — My Own Question

### Question

[Write one different, specific question about the SAME assigned county that the existing data can answer.]

### SQL

In [ ]:
sql_q3 = """
PASTE_YOUR_Q3_SQL_HERE
"""

### Python/Pandas

In [ ]:
q3 = read_query(sql_q3)
display(q3)

### Visualization

In [ ]:
# Plot q3 here. Gemini may help you write this code.
# Choose a chart that helps answer your Q3.

### Interpretation

[Write 1–2 sentences answering Q3 from your result/chart.]

## Final Summary

[In 1–2 sentences, state whether you completed Q1, Q2 and Q3 and created a visualization for each. Identify anything unfinished. Keep your findings under each question's Interpretation.]

## Gemini Use

[In 1–2 sentences, say whether Gemini was helpful, mention any issue you noticed, and say whether you checked its answer. A manual calculation is enough. If you did not use Gemini, say so.]

## Save and Submit

Save **lab6.sql** and **lab6.ipynb** at the root of your assigned private IA340 repository on **main**. Use **File → Save a copy to GitHub** for the notebook and reopen both files to confirm they are readable and outputs are visible.

Submit one repository-root URL in Canvas:

```text
IA340-1: https://github.com/JMU-Data/ia340-fa26-1-<your-github-username>
IA340-2: https://github.com/JMU-Data/ia340-fa26-2-<your-github-username>
```

Use your actual assigned repository. Do not submit a `/blob/main/...` file link, Colab URL, database IP, screenshots, or a separate report.

In [ ]:
cur.close()
conn.close()